# Volumen util de embalses diario

In [0]:
!pip install pydataxm

### API SIMEM Volumen de embalses ingestion y guardar json en raw files landing

In [0]:
from pydataxm.pydatasimem import ReadSIMEM, CatalogSIMEM

In [0]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

from pydataxm.pydatasimem import ReadSIMEM


bronze_table = "observatorio_dev.bronze.niveles_embalses"
historical_start_date = date(2026, 1, 1)
lookback_days = 45

fecha_fin = datetime.now(
    ZoneInfo("America/Bogota")
).date()

bronze_is_empty = len(
    spark.table(bronze_table).head(1)
) == 0

if bronze_is_empty:
    fecha_inicio = historical_start_date
    execution_mode = "BACKFILL"
else:
    fecha_inicio = fecha_fin - timedelta(days=lookback_days)
    execution_mode = "INCREMENTAL"

fecha_inicio_str = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_str = fecha_fin.strftime("%Y-%m-%d")

print("Modo de ejecución:", execution_mode)
print("Tabla Bronze:", bronze_table)
print(
    f"Rango solicitado a SIMEM: "
    f"{fecha_inicio_str} a {fecha_fin_str}"
)

df_niveles = ReadSIMEM(
    "BD26DC",
    fecha_inicio_str,
    fecha_fin_str
).main(filter=False)

if df_niveles is None or df_niveles.empty:
    raise ValueError(
        "SIMEM no devolvió registros para el rango solicitado"
    )

print(f"Registros descargados: {len(df_niveles):,}")


In [0]:

df_niveles.to_json(
    "/Volumes/observatorio_dev/landing/raw_files/"
    "niveles_embalses_plantas.json",
    orient="records",
    lines=True,
    mode="w"
)